# DSPyGen — Getting Started Notebook

This notebook demonstrates the **DSPyGen Jupyter magic extension**, which provides
a suite of `%magic` and `%%magic` commands that make it easy to:

- Configure a DSPy language-model backend in one line
- Browse available modules and agents
- Define DSPy Signatures interactively
- Run any DSPyGen module without writing boilerplate
- Compose multi-step pipelines with a YAML DSL
- Inspect the LM call history

---

**Prerequisites**

```bash
pip install dspygen dspy-ai pyyaml ipython
```

Set your OpenAI API key before running:

```bash
export OPENAI_API_KEY=sk-...
```

## 1 — Load the Extension

Use IPython's built-in `%load_ext` magic to activate all DSPyGen commands.
A welcome banner listing available magics will be printed once the extension loads.

In [ ]:
%load_ext dspygen.jupyter.magic

## 2 — Configure the Language Model

`%dspygen_init` calls `init_dspy` from `dspygen.utils.dspy_tools` and wires up
`dspy.settings.lm`.  Pass any model name recognised by `dspy.OpenAI`.

For local inference via Ollama prefix the model name with `ollama/`:

```
%dspygen_init ollama/llama3
```

The LM instance is stored in `_dspygen_lm` for programmatic access.

In [ ]:
%dspygen_init gpt-4o-mini

## 3 — Browse Available Modules

`%dspygen_modules` scans the `dspygen.modules` package and renders an HTML table
in Jupyter, or a plain-text table in a terminal IPython session.  Each row shows
the short module name and its fully-qualified import path.

In [ ]:
%dspygen_modules

## 4 — Browse Available Agents

`%dspygen_agents` does the same for the `dspygen.agents` package.  Agents are
stateful FSM-based orchestrators that can be composed with DSPy modules.

In [ ]:
%dspygen_agents

## 5 — Define a DSPy Signature Interactively

The `%%dspygen_signature` cell magic compiles a class body into a real
`dspy.Signature` subclass and injects it into the notebook namespace so you
can reference it immediately in subsequent cells.

Fields follow the standard DSPy annotation pattern:

```
<name>: <type> = dspy.InputField(desc="...")
<name>: <type> = dspy.OutputField(desc="...")
```

You may also include a docstring to describe the task — DSPy uses it as the
instruction prompt for the Signature.

In [ ]:
%%dspygen_signature MySummarizer
"""Summarize a piece of text into a single concise sentence."""
text: str = dspy.InputField(desc="The source text to be summarized.")
summary: str = dspy.OutputField(desc="A one-sentence summary of the text.")

In [ ]:
# Verify the signature was injected into the notebook namespace
import dspy

print("Class:", MySummarizer)
print("Fields:", list(MySummarizer.fields.keys()))

# Use the signature with ChainOfThought
predictor = dspy.ChainOfThought(MySummarizer)
print("Predictor created:", predictor)

## 6 — Run a DSPyGen Module

`%dspygen <module_name> [args]` dynamically imports the module from
`dspygen.modules`, finds the first `dspy.Module` subclass (or a top-level
`module_call` function), and invokes it.

- **Positional strings** are passed to `.forward()` in order
- **`key=value` pairs** become keyword arguments
- The return value is stored in `_dspygen_result` for re-use

The module name is resolved against both `dspygen.modules.<name>` and
`dspygen.modules.<name>_module` automatically.

In [ ]:
%dspygen text_summarizer_module "summarize this"

In [ ]:
# The result of the last %dspygen call is always accessible here
print("Result type:", type(_dspygen_result))
print("Result value:", _dspygen_result)

## 7 — YAML Pipeline

The `%%dspygen_pipeline` cell magic chains modules together using a lightweight
YAML DSL.  Each step specifies a `module`, optional `name`, and optional `args`.

The special placeholder `{{ previous_output }}` inserts the string representation
of the previous step's output, enabling basic data flow without Python glue code.

After execution the full results list is stored in `_dspygen_pipeline_results`.

In [ ]:
%%dspygen_pipeline
steps:
  - name: summarize
    module: text_summarizer_module
    args:
      text: >
        Artificial intelligence is transforming every industry.  Machine learning
        models trained on vast datasets can now perform tasks that previously
        required human expertise, from diagnosing diseases to writing software.
  - name: write_blog
    module: blog_module
    args:
      subject: "{{ previous_output }}"

In [ ]:
# Inspect all pipeline step results programmatically
for step in _dspygen_pipeline_results:
    print(f"Step  : {step['step']}")
    print(f"Module: {step['module']}")
    error = step.get('error')
    if error:
        print(f"ERROR : {error}")
    else:
        preview = str(step['result'])[:200] if step['result'] is not None else 'None'
        print(f"Output: {preview}")
    print()

## 8 — LM Call History

`%dspygen_history` displays every prompt/completion pair recorded by the
currently configured DSPy LM.  Pass an optional integer to limit the output
to the last *N* entries.

```
%dspygen_history       # show all recorded calls
%dspygen_history 5    # show the last 5 calls only
```

In [ ]:
%dspygen_history

## 9 — Keyword-argument Modules

You can pass keyword arguments directly on the magic line using `key=value`
syntax.  Quoted values and bare identifiers are both accepted.

Here we call `blog_module` with a named `subject` argument.

In [ ]:
%dspygen blog_module subject="The future of AI-assisted coding"

## 10 — Switching Models and Cleanup

You can call `%dspygen_init` multiple times to hot-swap the LM without
restarting the kernel — each call reconfigures `dspy.settings.lm` in place.

When finished, unload the extension with IPython's built-in `%unload_ext`.
This deregisters all `%dspygen_*` magics from the current session.

In [ ]:
# Hot-swap to a more capable model for a demanding task
%dspygen_init gpt-4o

# Run a quick sanity check with the new model
%dspygen blog_module subject="DSPy makes LLMs composable and optimisable"

In [ ]:
# Cleanly remove all DSPyGen magic commands from this session
%unload_ext dspygen.jupyter.magic